# 1. Configuración y carga de datos

# 1.1. Importación de librerías y funciones

Importamos las librerías necesarias para el modelado y la evaluación, incluyendo:
- `scanpy` y `pandas` para la manipulación de datos.
 - `sklearn` para el modelado (RandomForest, train_test_split, métricas).
- `joblib` para guardar nuestros modelos entrenados.
 - Nuestra función de ploteo personalizada desde la carpeta `src`.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

# Librerías de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sys.path.append('../src')
from models import train_and_evaluate_model
from plotting import plot_confusion_matrix
from counts_to_tpm import counts_to_tpm

# 1.2 Cargar datos de referencia

In [ ]:
DATA_PROCESSED_PATH = '../data/processed/'
PROCESSED_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
OUTPUTS_PATH = '../outputs/'
FIGURES_PATH = os.path.join(OUTPUTS_PATH, 'figures')
MODELS_PATH = os.path.join(OUTPUTS_PATH, 'models')

os.makedirs(FIGURES_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

adata_final = sc.read_h5ad(os.path.join(DATA_PROCESSED_PATH, PROCESSED_FILENAME))

print("Dataset procesado cargado exitosamente:")
print(adata_final)

# 2. Estrategia 1: Firma basada en marcadores específicos por clase

## 2.1. Identificar marcadores

Nuestra primera estrategia para construir una firma es identificar los genes más diferencialmente expresados para cada tipo celular de forma independiente.

In [ ]:
print("--- Ejecutando rank_genes_groups para obtener marcadores específicos ---")

# Verificamos si la capa 'log_normalized' existe, si no, la creamos
if 'log_normalized' not in adata_final.layers:
    print("Capa 'log_normalized' no encontrada, creándola desde .X")
    adata_final.layers['log_normalized'] = adata_final.X

sc.tl.rank_genes_groups(adata_final, groupby='cell_type', use_raw=False, layer='log_normalized', method='t-test', key_added='rank_genes_groups_specific')
marker_genes_df_specific = pd.DataFrame(adata_final.uns['rank_genes_groups_specific']['names'])

## 2.2. Construir la lista de genes

In [ ]:
print("\n--- Creando la matriz de firmas de 'Top N' marcadores ---")
n_specific_genes_per_class = 25
specific_genes_list = []
for col in marker_genes_df_specific.columns:
    specific_genes_list.extend(marker_genes_df_specific[col].head(n_specific_genes_per_class))
specific_genes_list = sorted(list(set(specific_genes_list)))

## 2.3. Cálculo de la matriz de firmas a partir de datos TPM

Para construir la matriz de firmas, primero normalizamos los conteos crudos de nuestro dataset de referencia a TPM. Luego, calculamos la expresión promedio para cada tipo celular.

In [ ]:
print("\n--- Calculando la matriz de firmas en escala TPM ---")

#Obtener los componentes necesarios desde el objeto AnnData de referencia
if adata_final.raw is None:
    raise ValueError("El objeto AnnData no tiene una capa `.raw`. Se necesitan los conteos crudos.")

gene_lengths = pd.to_numeric(adata_final.raw.var['feature_length'])
ref_counts_matrix = adata_final.raw.X
ref_gene_names = adata_final.raw.var_names
ref_cell_types = adata_final.obs['cell_type']

#Alinear los genes entre la matriz de conteos y las longitudes
common_genes = ref_gene_names.intersection(gene_lengths.index)
ref_counts_matrix_aligned = ref_counts_matrix[:, ref_gene_names.get_indexer(common_genes)].copy()
gene_lengths_aligned = gene_lengths[common_genes]

# Normalizar los conteos alineados a TPM
ref_tpm_matrix = counts_to_tpm(ref_counts_matrix_aligned, gene_lengths_aligned)

#Convertir la matriz TPM dispersa a un DataFrame de pandas para la agregación
ref_tpm_df = pd.DataFrame.sparse.from_spmatrix(
    ref_tpm_matrix,
    index=adata_final.obs.index,
    columns=common_genes
)
ref_tpm_df['cell_type'] = ref_cell_types.values

#Calcular la expresión promedio para crear la matriz de firmas completa
signature_matrix_full_tpm = ref_tpm_df.groupby('cell_type').mean().T

print("Matriz de firmas TPM completa calculada.")
print("Dimensiones:", signature_matrix_full_tpm.shape)
display(signature_matrix_full_tpm.head())

In [ ]:
# Filtramos por la lista de genes específicos
signature_matrix_topN = signature_matrix_full_tpm.loc[
    signature_matrix_full_tpm.index.isin(specific_genes_list)
]

## 2.3. Generar y guardar la matriz de firmas

In [ ]:
signature_topN_to_save = signature_matrix_topN.copy()
signature_topN_to_save.index.name = 'gene'
SIGNATURE_TOPN_FILENAME = 'signature_matrix_topN.tsv'
signature_topN_path = os.path.join(DATA_PROCESSED_PATH, SIGNATURE_TOPN_FILENAME)
signature_topN_to_save.to_csv(signature_topN_path, sep='\t')
print(f"Matriz de firmas de 'Top N' guardada en: {signature_topN_path}")

# 3. Estrategia 2: Firma basada en importancia global (Enfoque Híbrido)

## 3.1. Obtener genes de importancia global

Para una mayor robustez, identificamos los genes que son globalmente más importantes para distinguir entre todas las clases simultáneamente.

In [ ]:
print("\n--- Entrenando RF para obtener importancia global de genes ---")
X_full = adata_final.layers.get('log_normalized', adata_final.X)
y_full = adata_final.obs['cell_type']

feature_selection_rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
feature_selection_rf.fit(X_full, y_full)

feature_importance_df = pd.DataFrame({
    'gene': adata_final.var.index,
    'importance': feature_selection_rf.feature_importances_
}).sort_values('importance', ascending=False)

n_global_genes = 250
global_genes = feature_importance_df.head(n_global_genes)['gene'].tolist()

## 3.2. Crear y guardar la matriz de firmas híbrida

Combinamos los genes de importancia global con los marcadores más específicos de cada linaje para crear una firma final robusta y representativa.

In [ ]:
print("\n--- Creando la lista de genes híbrida ---")
hybrid_gene_set = set(global_genes) | set(specific_genes_list) # se usan los genes del apartado anterior
hybrid_gene_list = sorted(list(hybrid_gene_set))
print(f"La lista híbrida final contiene: {len(hybrid_gene_list)} genes únicos.")

In [ ]:
# Filtramos la matriz de firmas TPM completa por la lista híbrida
signature_matrix_hybrid = signature_matrix_full_tpm.loc[
    signature_matrix_full_tpm.index.isin(hybrid_gene_list)
]

In [ ]:
# Guardamos en formato TSV para R
signature_hybrid_to_save = signature_matrix_hybrid.copy()
signature_hybrid_to_save.index.name = 'gene'
HYBRID_SIGNATURE_FILENAME = 'hybrid_signature_matrix.tsv'
hybrid_signature_path = os.path.join(DATA_PROCESSED_PATH, HYBRID_SIGNATURE_FILENAME)
signature_hybrid_to_save.to_csv(hybrid_signature_path, sep='\t')
print(f"Matriz de firmas híbrida guardada en: {hybrid_signature_path}")

# 4. Comparación Cuantitativa de las Estrategias de Selección de Genes

# 4.1. Justificación

Hemos generado tres conjuntos de genes candidatos para nuestra firma:
1. Específicos: Top 25 marcadores por clase.
2. Globales: Top 250 genes según la importancia del RF.
3. Híbridos: La unión de los dos anteriores.

Para determinar objetivamente qué conjunto es más informativo, compararemos el rendimiento de un clasificador Random Forest entrenado con cada uno de ellos.

In [ ]:
from sklearn.metrics import f1_score

# Diccionario para guardar los resultados
results = {}

# Listas de genes
gene_lists = {
    "Específicos (Top 25/clase)": specific_genes_list,
    "Globales (RF Top 250)": global_genes,
    "Híbridos": hybrid_gene_list
}

y = adata_final.obs['cell_type']

for name, gene_list in gene_lists.items():
    print(f"\n--- Evaluando la firma: {name} ({len(gene_list)} genes) ---")
    
    # Preparar datos
    X = adata_final[:, gene_list].X
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Entrenar modelo
    model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    # Evaluar
    y_pred = model.predict(X_test)
    
    # Guardar métricas
    results[name] = {
        'n_genes': len(gene_list),
        'macro_f1': f1_score(y_test, y_pred, average='macro'),
        'f1_epithelial': f1_score(y_test, y_pred, labels=['epithelial cell'], average='macro'),
        'f1_pdc': f1_score(y_test, y_pred, labels=['plasmacytoid dendritic cell'], average='macro')
    }

## 4.2. Conclusión de la Comparación de Firmas

In [ ]:
results_df = pd.DataFrame(results).T
print("\n--- Tabla Comparativa de Rendimiento de Firmas ---")
display(results_df.sort_values('macro_f1', ascending=False))

# 5. Validación de la calidad de los genes seleccionados

## 5.1. Benchmark de clasificadores con la firma de genes específicos

Para validar la calidad de nuestro conjunto de genes seleccionados, realizamos un benchmark de clasificadores (RF, XGBoost, MLP) utilizando únicamente este subconjunto de genes como características.

In [ ]:
print("\n--- Iniciando benchmark de clasificadores con genes específicos ---")

# Preparamos los datos de entrada para el benchmark
X_specific = adata_final[:, specific_genes_list].X
y_specific = adata_final.obs['cell_type']

X_train, X_test, y_train, y_test = train_test_split(
    X_specific, y_specific, test_size=0.2, random_state=42, stratify=y_specific
)

print(f"Datos divididos en entrenamiento ({X_train.shape[0]} células) y prueba ({X_test.shape[0]} células).")
print(f"Número de características (genes): {X_train.shape[1]}")

## 5.1.1 Random Forest

In [ ]:
print("\n--- Iniciando Benchmark 1: Random Forest ---")

model_rf, report_rf, cm_rf, classes_rf = train_and_evaluate_model(
    'rf',
    X_train, y_train,
    X_test, y_test,
    output_dir=OUTPUTS_PATH,
    model_name='rf_specific_genes'
)
print("\nReporte de Clasificación (Random Forest):")
print(report_rf)
plot_confusion_matrix(
    cm_rf, classes_rf, 
    title='Matriz de Confusión - RF (Genes específicos)', 
    save_path=os.path.join(FIGURES_PATH, 'cm_rf_specific_genes.png')
)

## 5.1.2 XGBoost

In [ ]:
print("\n--- Iniciando Benchmark 2: XGBoost ---")

model_xgb, report_xgb, cm_xgb, classes_xgb = train_and_evaluate_model(
    'xgb',
    X_train, y_train,
    X_test, y_test,
    output_dir=OUTPUTS_PATH,
    model_name='xgb_specific_genes'
)
print("\nReporte de Clasificación (XGBoost):")
print(report_xgb)
plot_confusion_matrix(
    cm_xgb, classes_xgb, 
    title='Matriz de Confusión - XGBoost (Genes específicos)',
    save_path=os.path.join(FIGURES_PATH, 'cm_xgb_specific_genes.png')
)

## 5.1.3 MLP

In [ ]:
model_mlp, report_mlp, cm_mlp, classes_mlp = train_and_evaluate_model(
    'mlp',
    X_train, y_train,
    X_test, y_test,
    output_dir=OUTPUTS_PATH,
    model_name='mlp_specific_genes',
    hidden_layer_sizes=(128, 64, 32),  #Ajustar??
    max_iter=500
)
print("\nReporte de Clasificación (MLP):")
print(report_mlp)
plot_confusion_matrix(
    cm_mlp, classes_mlp, 
    title='Matriz de Confusión - MLP (Genes específicos)',
    save_path=os.path.join(FIGURES_PATH, 'cm_mlp_specific_genes.png')
)

# 6. Conclusión de la selección de firmas y validación final

## 6.1. Selección de la estrategia óptima para la selección de genes

Tras comparar tres estrategias diferentes para la selección de genes (basada en marcadores específicos por clase, en importancia global, y una combinación híbrida), el análisis cuantitativo demostró que el conjunto de genes compuesto por los **25 mejores marcadores específicos por cada tipo celular** proporcionaba el rendimiento de clasificación más alto y equilibrado. Esta firma, con un total de 211 genes únicos, fue seleccionada como la base para la construcción de la matriz de firmas final.

## 6.2. Validación final de la calidad de la firma mediante benchmark de clasificadores

Para validar el poder discriminativo de este conjunto de genes seleccionado, se realizó un benchmark final comparando tres algoritmos de clasificación: Random Forest (RF), XGBoost y un Perceptrón Multicapa (MLP).

### Tabla comparativa de rendimiento (firma de genes específicos)

| Métrica / Modelo                | Random Forest (RF) | XGBoost (XGB)     | MLP (Red Neuronal) |
| :------------------------------ | :----------------: | :---------------: | :----------------: |
| **Precisión (Accuracy)**        | 96.45%             | **96.52%**        | **96.52%**         |
| **Macro Avg F1-Score**          | 0.93               | **0.94**          | **0.94**           |
| **F1-Score `epithelial cell`**  | 0.56               | **0.60**          | 0.58               |
| **F1-Score `pDC`**              | 0.95               | 0.96              | **0.97**           |

### Veredicto del benchmark

Los resultados del benchmark confirman la alta calidad de la firma de genes específicos. Los tres modelos alcanzan un rendimiento muy elevado, pero XGBoost se puede considerar como el clasificador sutilmente superior, logrando el mejor F1-Score en la clase más desafiante (`epithelial cell`) y empatando en las métricas generales más altas.

El buen rendimiento de este benchmark sirve como una validación final y robusta del pipeline de selección de características y de la calidad del dataset de referencia. Se ha demostrado que es posible distinguir los tipos celulares de interés con una precisión muy alta, esto da una gran confianza en la matriz de firmas que se utilizará para la deconvolución.

## 6.3. Archivos de salida para la deconvolución

Este notebook ha generado el siguiente archivo clave, que será el input para el pipeline de deconvolución en R:

-   **`signature_matrix_specific_markers.tsv`**: La matriz de firmas final, construida a partir de la expresión promedio en escala TPM de los 211 genes marcadores específicos.